# ESC-50 — mel-CNN training for CV Studio AudioClassification

Trains a mel-spectrogram CNN on the [ESC-50](https://github.com/karoldvl/ESC-50) dataset
(50 environmental sound classes, 5 s clips).

Exports an ONNX model **directly compatible** with the CV Studio `AudioClassification` node:
- Input shape `(1, 1, 128, 216)` — `(batch, channels, n_mels, time)`
- Output shape `(1, 50)` — softmax class scores
- ONNX metadata key `names` → JSON class-name dict

Upload the exported `.onnx` via the **📂 Add Model** button in the AudioClassification node.

> **Run on Google Colab** (free GPU): `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio librosa onnx onnxruntime pandas scikit-learn tqdm


In [ ]:
import os, json, zipfile, urllib.request, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import onnx
import onnxruntime as ort
from tqdm.auto import tqdm

# ── Hyper-parameters (MUST match CV Studio AudioClassification node defaults) ──
SR          = 22_050   # sample rate
N_MELS      = 128      # mel bands
N_FFT       = 2_048    # STFT window size
HOP_LENGTH  = 512      # STFT hop
MAX_SEC     = 5        # clip duration
N_CLASSES   = 50

# Derived: number of time frames for a MAX_SEC clip (center=True librosa default)
T_FRAMES = 1 + int(SR * MAX_SEC) // HOP_LENGTH  # 216
print(f'Mel shape per clip: ({N_MELS}, {T_FRAMES})')  # (128, 216)

BATCH_SIZE  = 32
EPOCHS      = 60
LR          = 1e-3
VAL_FOLD    = 5       # ESC-50 cross-validation fold used for validation
ONNX_PATH   = 'esc50_cvstudio.onnx'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
ESC50_CLASS_NAMES = {
    0: 'Dog', 1: 'Rooster', 2: 'Pig', 3: 'Cow', 4: 'Frog',
    5: 'Cat', 6: 'Hen', 7: 'Insects (flying)', 8: 'Sheep', 9: 'Crow',
    10: 'Rain', 11: 'Sea waves', 12: 'Crackling fire', 13: 'Crickets',
    14: 'Chirping birds', 15: 'Water drops', 16: 'Wind',
    17: 'Pouring water', 18: 'Toilet flush', 19: 'Thunderstorm',
    20: 'Crying baby', 21: 'Sneezing', 22: 'Clapping',
    23: 'Breathing', 24: 'Coughing', 25: 'Footsteps',
    26: 'Laughing', 27: 'Brushing teeth', 28: 'Snoring',
    29: 'Drinking (sipping)', 30: 'Door knock', 31: 'Mouse click',
    32: 'Keyboard typing', 33: 'Door, wood creaks', 34: 'Can opening',
    35: 'Washing machine', 36: 'Vacuum cleaner', 37: 'Clock alarm',
    38: 'Clock tick', 39: 'Glass breaking', 40: 'Helicopter',
    41: 'Chainsaw', 42: 'Siren', 43: 'Car horn', 44: 'Engine',
    45: 'Train', 46: 'Church bells', 47: 'Airplane',
    48: 'Fireworks', 49: 'Hand saw',
}


In [ ]:
DATASET_ROOT = 'ESC-50-master'
AUDIO_DIR    = os.path.join(DATASET_ROOT, 'audio')
CSV_PATH     = os.path.join(DATASET_ROOT, 'meta', 'esc50.csv')

if not os.path.isdir(DATASET_ROOT):
    print('Downloading ESC-50 …')
    url = 'https://github.com/karoldvl/ESC-50/archive/master.zip'
    urllib.request.urlretrieve(url, 'ESC-50.zip')
    with zipfile.ZipFile('ESC-50.zip') as z:
        z.extractall('.')
    os.remove('ESC-50.zip')
    print('Done.')
else:
    print('ESC-50 already present.')


In [ ]:
class ESC50Dataset(Dataset):
    """Mel-spectrogram dataset for ESC-50."""

    def __init__(self, df, audio_dir, augment=False):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.augment   = augment
        self.max_len   = SR * MAX_SEC

    def __len__(self):
        return len(self.df)

    def _load_mel(self, path):
        y, _ = librosa.load(path, sr=SR, mono=True)
        # Pad / crop to MAX_SEC
        if len(y) < self.max_len:
            y = np.pad(y, (0, self.max_len - len(y)))
        else:
            y = y[:self.max_len]
        # Optional time-shift augmentation
        if self.augment and random.random() < 0.5:
            shift = random.randint(-SR // 2, SR // 2)
            y = np.roll(y, shift)
        mel    = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db = librosa.power_to_db(mel).astype(np.float32)
        return mel_db  # (N_MELS, T_FRAMES)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.audio_dir, row['filename'])
        mel   = torch.tensor(self._load_mel(path)).unsqueeze(0)  # (1, N_MELS, T)
        label = int(row['target'])
        return mel, label


df = pd.read_csv(CSV_PATH)
train_df = df[df['fold'] != VAL_FOLD]
val_df   = df[df['fold'] == VAL_FOLD]

train_ds = ESC50Dataset(train_df, AUDIO_DIR, augment=True)
val_ds   = ESC50Dataset(val_df,   AUDIO_DIR, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} clips | Val: {len(val_ds)} clips')


In [ ]:
class MelCNN(nn.Module):
    """Lightweight CNN for mel-spectrogram classification."""

    def __init__(self, n_mels=N_MELS, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))  # raw logits


model = MelCNN().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total_params:,}')


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    total_loss = correct = total = 0
    for mels, labels in train_loader:
        mels, labels = mels.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(mels), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * mels.size(0)
        correct    += (model(mels).argmax(1) == labels).sum().item()
        total      += mels.size(0)
    scheduler.step()

    # ── Validate ──
    model.eval()
    v_correct = v_total = 0
    with torch.no_grad():
        for mels, labels in val_loader:
            mels, labels = mels.to(DEVICE), labels.to(DEVICE)
            v_correct += (model(mels).argmax(1) == labels).sum().item()
            v_total   += mels.size(0)

    val_acc = v_correct / v_total
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'esc50_best.pth')

    if epoch % 10 == 0:
        print(f'Ep {epoch:3d} | loss {total_loss/total:.4f} | val {val_acc:.4f} | best {best_acc:.4f}')

print(f'\nBest validation accuracy: {best_acc:.4f}')


In [ ]:
# ── Reload best weights ──
model.load_state_dict(torch.load('esc50_best.pth', map_location='cpu'))
model.eval()

# Wrap with softmax for ONNX export
class _WithSoftmax(nn.Module):
    def __init__(self, base): super().__init__(); self.base = base; self.sm = nn.Softmax(dim=1)
    def forward(self, x): return self.sm(self.base(x))

export_model = _WithSoftmax(model).eval()

# Fixed input shape: (1, 1, N_MELS, T_FRAMES)
dummy = torch.zeros(1, 1, N_MELS, T_FRAMES)

torch.onnx.export(
    export_model, dummy, ONNX_PATH,
    opset_version=13,
    input_names=['mel_input'],
    output_names=['class_scores'],
    dynamic_axes=None,  # fixed shape — no dynamic axes
)
print(f'Exported (without metadata): {ONNX_PATH}')

# ── Embed class names into ONNX metadata ──
# CV Studio reads the 'names' key (JSON dict string) at upload time.
proto = onnx.load(ONNX_PATH)
m = proto.metadata_props.add()
m.key   = 'names'
m.value = json.dumps({str(k): v for k, v in ESC50_CLASS_NAMES.items()})
onnx.save(proto, ONNX_PATH)
print(f'Class names embedded → {ONNX_PATH}')


In [ ]:
# ── Verify ONNX model ──
sess      = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp_shape = sess.get_inputs()[0].shape
out_shape = sess.get_outputs()[0].shape
meta      = sess.get_modelmeta().custom_metadata_map
names     = json.loads(meta.get('names', '{}'))

print('Input shape  :', inp_shape)   # [1, 1, 128, 216]
print('Output shape :', out_shape)   # [1, 50]
print('Class names  :', len(names), 'classes embedded')
print('First 5 :', {k: names[k] for k in list(names)[:5]})

# Quick inference test
dummy_np = np.zeros((1, 1, N_MELS, T_FRAMES), dtype=np.float32)
scores   = sess.run(None, {'mel_input': dummy_np})[0]
top_idx  = int(np.argmax(scores))
print(f'\nTest inference OK — top class: {names[str(top_idx)]} ({scores[0, top_idx]:.4f})')
print(f'\n✓ Upload "{ONNX_PATH}" via the 📂 Add Model button in CV Studio!')
